# Cross-dataset generalization with deep-learning models, round 2: strict text cleaning (Kaggle)

**Why a second round:** in round 1 the `delaked` text kept punctuation, so leftovers like `washington ( ) -` could still reveal the source. This notebook adds a `strict` mode (dateline and all punctuation removed) and repeats the key models with 3 random seeds.

**Setup (right-hand panel):** add the two datasets (Fake and Real News Dataset; Fake News Classification), Settings -> Accelerator -> GPU, Settings -> Internet -> On. Run cells top to bottom.

## 1. Check the GPU and locate the data

In [ ]:
import glob, torch
print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found. Did you add the dataset in the right-hand panel?"
    return hits[0]

FAKE, TRUE, WEL = find("Fake.csv"), find("True.csv"), find("WELFake_Dataset.csv")
print(FAKE); print(TRUE); print(WEL)

## 2. Build the LIAR file

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("UKPLab/liar")
df = pd.concat([ds[s].to_pandas() for s in ds.keys()], ignore_index=True)
df["label"] = df["label_text"].astype(str).str.lower().map({"true statement": 1, "false statement": 0})
assert df["label"].notna().all(), "unexpected label values"
df[["text", "label"]].to_csv("/kaggle/working/liar_binary.csv", index=False)
print(len(df), df["label"].mean())

## 3. Write the two scripts

In [ ]:
%%writefile /kaggle/working/phase7_cross_dataset.py
"""
phase7_cross_dataset.py
=======================
Cross-dataset generalization experiment for the Fake News Detector project.

Question answered: "I trained on ISOT and got ~97%. What happens when the SAME
pipeline (TF-IDF 1-2 grams / 5000 features -> LinearSVC etc.) is trained on one
dataset and tested on a dataset it has never seen?"

It reuses your final pipeline from phase6_rebuild_corrected_v2.py
(strip URLs -> remove leakage terms -> standard clean) and compares it against
the basic phase2 cleaning, so you also get a leakage ablation for free:

    mode "basic"   = phase2 cleaning only (lowercase, letters only, stopwords)
    mode "delaked" = your final pipeline (URLs + Reuters/wire/day/month terms removed)

Outputs (in --out-dir):
    results_<mode>.csv            long-form table, every (train, test, model, seed)
    matrix_<mode>_<model>.png     heatmap, rows = trained on, cols = tested on
    (and printed accuracy / F1 tables with mean +- std over seeds)

Usage examples
--------------
# ISOT + WELFake + LIAR (any subset of >= 2 datasets works)
python phase7_cross_dataset.py \
    --isot-fake Fake.csv --isot-true True.csv \
    --welfake WELFake_Dataset.csv \
    --liar-dir liar_dataset/          # folder holding train.tsv valid.tsv test.tsv

# quick smoke run on a subset
python phase7_cross_dataset.py ... --max-per-dataset 5000 --seeds 42,43

Label convention (same as your project): 1 = True/real, 0 = Fake.
"""
from __future__ import annotations

import argparse
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# --------------------------------------------------------------------------
# Cleaning (copied from your phase6_rebuild_corrected_v2.py so results match)
# --------------------------------------------------------------------------
LEAKAGE_PHRASES = [
    "washington reuters", "told reuters", "told reporters", "featured image",
    "image via", "getty images", "twitter com", "daily mail", "reuters", "said",
    "via", "read", "featured", "image", "getty", "images", "com", "wire",
    "watch", "ap", "us", "gop", "hillary", "mr", "obama", "rep", "sen",
]
DAY_WORDS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
MONTH_WORDS = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "sept", "oct", "nov", "dec"]
URL_PATTERN = re.compile(
    r"(?i)\b(?:https?://|www\.)\S+\b|\b[a-z0-9-]+(?:\.[a-z0-9-]+)+(?:/\S*)?\b"
)

_PHRASE_PATTERNS = [
    re.compile(rf"\b{re.escape(p)}\b")
    for p in sorted(set(LEAKAGE_PHRASES), key=len, reverse=True)
]
_DAY_PATTERN = re.compile(r"\b(" + "|".join(map(re.escape, DAY_WORDS)) + r")\b")
_MONTH_PATTERN = re.compile(r"\b(" + "|".join(map(re.escape, MONTH_WORDS)) + r")\b")


def _load_stop_words() -> set[str]:
    try:
        import nltk
        from nltk.corpus import stopwords

        try:
            return set(stopwords.words("english"))
        except LookupError:
            nltk.download("stopwords", quiet=True)
            return set(stopwords.words("english"))
    except Exception:  # offline / nltk missing
        print("[warn] NLTK stopwords unavailable, falling back to sklearn's stop-word list "
              "(numbers will differ very slightly from your README).")
        return set(ENGLISH_STOP_WORDS)


STOP_WORDS = _load_stop_words()


def strip_urls(text: str) -> str:
    cleaned = str(text).lower()
    cleaned = URL_PATTERN.sub(" ", cleaned)
    cleaned = re.sub(r"\b(?:http|https|www|youtube|youtu|com|org|net|edu|gov|bitly|bit\.ly)\b", " ", cleaned)
    return re.sub(r"\s+", " ", cleaned).strip()


def remove_leakage_terms(text: str) -> str:
    cleaned = str(text).lower()
    for pat in _PHRASE_PATTERNS:
        cleaned = pat.sub(" ", cleaned)
    cleaned = _DAY_PATTERN.sub(" ", cleaned)
    cleaned = _MONTH_PATTERN.sub(" ", cleaned)
    return re.sub(r"\s+", " ", cleaned).strip()


def standard_clean(text: str) -> str:
    cleaned = re.sub(r"[^a-z\s]", " ", text.lower())
    tokens = [t for t in cleaned.split()
              if t not in STOP_WORDS and not re.search(r"(http|www|youtube)", t)]
    return " ".join(tokens)


def basic_clean(text: str) -> str:
    """phase2: lowercase, letters only, stop-words removed."""
    cleaned = re.sub(r"[^a-z\s]", " ", str(text).lower())
    return " ".join(t for t in cleaned.split() if t not in STOP_WORDS)


def delaked_clean(text: str) -> str:
    """Your final pipeline (phase6_rebuild_corrected_v2.rebuild_clean_text)."""
    return standard_clean(remove_leakage_terms(strip_urls(text)))


def real_word_count(text: str) -> int:
    cleaned = re.sub(r"[^a-z\s]", " ", strip_urls(text))
    return len(cleaned.split())


CLEANERS = {"basic": basic_clean, "delaked": delaked_clean}

# --------------------------------------------------------------------------
# Dataset loaders -> DataFrame[text, label]  (1 = True/real, 0 = Fake)
# --------------------------------------------------------------------------

def _read_csv(path: Path, **kw) -> pd.DataFrame:
    try:
        return pd.read_csv(path, **kw)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp1252", **kw)


def load_isot(fake_path: Path, true_path: Path) -> pd.DataFrame:
    fake = _read_csv(fake_path)
    true = _read_csv(true_path)
    fake["label"], true["label"] = 0, 1
    return pd.concat([true[["text", "label"]], fake[["text", "label"]]], ignore_index=True)


def load_welfake(path: Path, flip: bool) -> pd.DataFrame:
    df = _read_csv(path)
    if "text" not in df.columns or "label" not in df.columns:
        raise SystemExit(f"WELFake file needs 'text' and 'label' columns, found {list(df.columns)}")
    df = df[["text", "label"]].copy()
    df["label"] = df["label"].astype(int)
    # NOTE: WELFake's documentation is inconsistent about which value means 'fake'.
    # Default here: 1 = real, 0 = fake (matches the paper). Use --welfake-flip if
    # the sanity check at the end warns about systematically below-chance accuracy.
    if flip:
        df["label"] = 1 - df["label"]
    return df


LIAR_COLS = ["id", "label", "text", "subject", "speaker", "job", "state", "party",
             "c_barely_true", "c_false", "c_half_true", "c_mostly_true", "c_pants_fire", "context"]
LIAR_MAP = {"true": 1, "mostly-true": 1, "false": 0, "barely-true": 0, "pants-fire": 0}  # half-true dropped


def load_liar(folder: Path) -> pd.DataFrame:
    frames = []
    for name in ("train.tsv", "valid.tsv", "test.tsv"):
        p = folder / name
        if p.exists():
            frames.append(pd.read_csv(p, sep="\t", header=None, names=LIAR_COLS, quoting=3))
    if not frames:
        raise SystemExit(f"No train.tsv / valid.tsv / test.tsv found in {folder}")
    df = pd.concat(frames, ignore_index=True)
    df["label"] = df["label"].map(LIAR_MAP)  # 'half-true' -> NaN -> dropped
    return df[["text", "label"]].dropna(subset=["label"]).astype({"label": int})


def load_liar_csv(path: Path) -> pd.DataFrame:
    """Binary LIAR exported as a CSV with columns text,label (1 = true, 0 = false)."""
    df = _read_csv(path)
    if not {"text", "label"}.issubset(df.columns):
        raise SystemExit(f"--liar-csv needs 'text' and 'label' columns, found {list(df.columns)}")
    return df[["text", "label"]].dropna(subset=["label"]).astype({"label": int})


MIN_WORDS = {"isot": 20, "welfake": 20, "liar": 5}  # LIAR statements are one-liners


def finalize(df: pd.DataFrame, name: str) -> pd.DataFrame:
    df = df.copy()
    df["text"] = df["text"].fillna("").astype(str)
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
    df = df[df["text"].map(real_word_count) >= MIN_WORDS[name]].reset_index(drop=True)
    return df


def subsample(df: pd.DataFrame, max_n: int | None) -> pd.DataFrame:
    if not max_n or len(df) <= max_n:
        return df
    parts = [g.sample(int(round(max_n * len(g) / len(df))), random_state=0) for _, g in df.groupby("label")]
    return pd.concat(parts, ignore_index=True)


def dedup_across(raw: dict[str, pd.DataFrame]) -> None:
    """Remove from each dataset every article already present in an EARLIER dataset
    (priority = order loaded: ISOT, WELFake, LIAR). Done BEFORE any train/test split, so
    no article can sit in both the training side and the test side of a cross-dataset pair.
    Also prints how often the duplicated articles carry the same label in both datasets."""
    seen: dict[str, int] = {}
    for name in list(raw):
        df = raw[name]
        key = df["text"].map(basic_clean)
        dup = key.isin(seen)
        if dup.any():
            same = (df.loc[dup, "label"].to_numpy() == key[dup].map(seen).to_numpy()).mean()
            verdict = "labels consistent" if same > 0.9 else ("LABELS LOOK INVERTED -> try --welfake-flip" if same < 0.1 else "mixed")
            print(f"[dedup] {name}: {int(dup.sum())} of {len(df)} articles already in an earlier dataset "
                  f"({dup.mean()*100:.1f}%); their labels agree {same*100:.1f}% -> {verdict}")
        else:
            print(f"[dedup] {name}: no articles shared with earlier datasets")
        kept = df[~dup].reset_index(drop=True)
        for k, l in zip(key[~dup], kept["label"]):
            seen[k] = int(l)
        raw[name] = kept


# --------------------------------------------------------------------------
# Experiment
# --------------------------------------------------------------------------
MODEL_FACTORIES = {
    "linear_svc": lambda seed: LinearSVC(),
    "logreg": lambda seed: LogisticRegression(max_iter=1000),
    "naive_bayes": lambda seed: MultinomialNB(),
}


def run(datasets: dict[str, pd.DataFrame], mode: str, models: list[str], seeds: list[int]) -> pd.DataFrame:
    names = list(datasets)
    rows = []
    for src in names:
        df = datasets[src]
        for seed in seeds:
            tr_idx, te_idx = train_test_split(
                np.arange(len(df)), test_size=0.2, random_state=seed, stratify=df["label"])
            train, own_test = df.iloc[tr_idx], df.iloc[te_idx]
            vec = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)   # fit on TRAIN ONLY
            Xtr = vec.fit_transform(train["clean"])
            train_set = set(train["clean"])
            fitted = {m: MODEL_FACTORIES[m](seed).fit(Xtr, train["label"]) for m in models}

            for tgt in names:
                if tgt == src:
                    test = own_test
                    overlap_removed = 0
                    overlap_agree = float("nan")
                else:
                    test = datasets[tgt]
                    keep = ~test["clean"].isin(train_set)   # drop exact duplicates seen in training
                    overlap_removed = int((~keep).sum())
                    if overlap_removed:
                        # do the two datasets give the SAME article the same label?
                        lab = dict(zip(train["clean"], train["label"]))
                        dup = test[~keep]
                        overlap_agree = float((dup["label"].to_numpy() == dup["clean"].map(lab).to_numpy()).mean())
                    else:
                        overlap_agree = float("nan")
                    test = test[keep]
                if len(test) == 0:
                    continue
                Xte = vec.transform(test["clean"])
                y = test["label"].to_numpy()
                majority = max(y.mean(), 1 - y.mean())
                for m, clf in fitted.items():
                    pred = clf.predict(Xte)
                    rows.append(dict(
                        mode=mode, train=src, test=tgt, model=m, seed=seed, n_test=len(test),
                        accuracy=accuracy_score(y, pred),
                        balanced_acc=balanced_accuracy_score(y, pred),
                        macro_f1=f1_score(y, pred, average="macro"),
                        pred_true_rate=float(pred.mean()), actual_true_rate=float(y.mean()),
                        majority_baseline=float(majority), overlap_removed=overlap_removed,
                        overlap_label_agreement=overlap_agree,
                    ))
    return pd.DataFrame(rows)


def pivot(res: pd.DataFrame, model: str, metric: str, names: list[str]):
    sub = res[res["model"] == model]
    g = sub.groupby(["train", "test"])[metric].agg(["mean", "std"]).reset_index()
    mean = g.pivot(index="train", columns="test", values="mean").reindex(index=names, columns=names)
    std = g.pivot(index="train", columns="test", values="std").reindex(index=names, columns=names)
    return mean, std


def fmt_table(mean: pd.DataFrame, std: pd.DataFrame) -> pd.DataFrame:
    out = mean.copy().astype(object)
    for r in mean.index:
        for c in mean.columns:
            m, s = mean.loc[r, c], std.loc[r, c]
            out.loc[r, c] = "-" if pd.isna(m) else f"{m*100:5.2f} +- {0 if pd.isna(s) else s*100:4.2f}"
    out.index.name = "trained on \\ tested on"
    return out


def heatmap(mean: pd.DataFrame, title: str, path: Path) -> None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(1.8 + 1.5 * len(mean.columns), 1.6 + 1.2 * len(mean)))
    im = ax.imshow(mean.values * 100, vmin=40, vmax=100, cmap="RdYlGn")
    ax.set_xticks(range(len(mean.columns)), mean.columns)
    ax.set_yticks(range(len(mean.index)), mean.index)
    ax.set_xlabel("tested on")
    ax.set_ylabel("trained on")
    ax.set_title(title)
    for i in range(mean.shape[0]):
        for j in range(mean.shape[1]):
            v = mean.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v*100:.1f}", ha="center", va="center", fontsize=11, fontweight="bold")
    fig.colorbar(im, ax=ax, label="accuracy (%)")
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--isot-fake", type=Path)
    ap.add_argument("--isot-true", type=Path)
    ap.add_argument("--welfake", type=Path, help="WELFake_Dataset.csv")
    ap.add_argument("--welfake-flip", action="store_true", help="invert WELFake labels")
    ap.add_argument("--liar-dir", type=Path, help="folder with train.tsv / valid.tsv / test.tsv")
    ap.add_argument("--liar-csv", type=Path, help="binary LIAR as CSV with columns text,label (1=true)")
    ap.add_argument("--no-dedup-across", action="store_true",
                    help="keep articles that appear in more than one dataset (NOT recommended)")
    ap.add_argument("--mode", choices=["basic", "delaked", "both"], default="both")
    ap.add_argument("--models", default="linear_svc,logreg,naive_bayes")
    ap.add_argument("--seeds", default="42,43,44,45,46")
    ap.add_argument("--max-per-dataset", type=int, default=None, help="stratified cap for quick runs")
    ap.add_argument("--out-dir", type=Path, default=Path("cross_dataset_results"))
    args = ap.parse_args()

    models = [m.strip() for m in args.models.split(",")]
    seeds = [int(s) for s in args.seeds.split(",")]
    args.out_dir.mkdir(parents=True, exist_ok=True)

    raw: dict[str, pd.DataFrame] = {}
    if args.isot_fake and args.isot_true:
        raw["ISOT"] = finalize(load_isot(args.isot_fake, args.isot_true), "isot")
    if args.welfake:
        raw["WELFake"] = finalize(load_welfake(args.welfake, args.welfake_flip), "welfake")
    if args.liar_dir:
        raw["LIAR"] = finalize(load_liar(args.liar_dir), "liar")
    elif args.liar_csv:
        raw["LIAR"] = finalize(load_liar_csv(args.liar_csv), "liar")
    if len(raw) < 2:
        raise SystemExit("Need at least 2 datasets (e.g. --isot-fake/--isot-true plus --welfake or --liar-dir).")

    if not args.no_dedup_across:
        print("\n=== Removing articles shared between datasets (before any split) ===")
        dedup_across(raw)
    raw = {n: subsample(d, args.max_per_dataset) for n, d in raw.items()}

    print("\n=== Datasets after dedup / length filter ===")
    for n, d in raw.items():
        print(f"{n:8s} rows={len(d):6d}  true={d['label'].mean()*100:5.1f}%  "
              f"median words={int(d['text'].str.split().str.len().median())}")

    names = list(raw)
    modes = ["basic", "delaked"] if args.mode == "both" else [args.mode]
    all_res = []
    for mode in modes:
        t0 = time.time()
        print(f"\n=== Cleaning ({mode}) ===")
        prepared = {}
        for n, d in raw.items():
            d = d.copy()
            d["clean"] = d["text"].map(CLEANERS[mode])
            prepared[n] = d[d["clean"].str.len() > 0].reset_index(drop=True)
        print(f"cleaned in {time.time()-t0:.0f}s; training / evaluating ...")
        res = run(prepared, mode, models, seeds)
        res.to_csv(args.out_dir / f"results_{mode}.csv", index=False)
        all_res.append(res)

        for m in models:
            acc_m, acc_s = pivot(res, m, "accuracy", names)
            f1_m, f1_s = pivot(res, m, "macro_f1", names)
            print(f"\n--- {mode} | {m} | accuracy % (mean +- std over {len(seeds)} seeds) ---")
            print(fmt_table(acc_m, acc_s).to_string())
            print(f"--- {mode} | {m} | macro-F1 % ---")
            print(fmt_table(f1_m, f1_s).to_string())
            heatmap(acc_m, f"{m} ({mode} cleaning)", args.out_dir / f"matrix_{mode}_{m}.png")

        # diagnostic: what does the primary model predict on foreign data?
        primary = models[0]
        diag = (res[(res["model"] == primary) & (res["train"] != res["test"])]
                .groupby(["train", "test"])[["accuracy", "balanced_acc", "majority_baseline", "pred_true_rate",
                                             "actual_true_rate", "overlap_removed",
                                             "overlap_label_agreement"]].mean().round(3))
        print(f"\n--- {mode} | {primary} | cross-dataset diagnostics "
              f"(acc vs majority baseline; share predicted 'True' vs actual) ---")
        print(diag.to_string())

    full = pd.concat(all_res, ignore_index=True)
    primary = models[0]
    print("\n=== Summary for", primary, "===")
    for mode in modes:
        s = full[(full["mode"] == mode) & (full["model"] == primary)]
        within = s[s["train"] == s["test"]]["accuracy"].mean() * 100
        cross = s[s["train"] != s["test"]]["accuracy"].mean() * 100
        print(f"{mode:8s} in-domain {within:5.2f}%   cross-dataset {cross:5.2f}%   gap {within-cross:5.2f} pts")

    # sanity check for inverted labels. A constant predictor scores 0.50 balanced accuracy,
    # so values well below 0.5 (or overlapping articles that carry OPPOSITE labels) point to
    # inverted labels - not just class imbalance.
    x = full[(full["model"] == primary) & (full["train"] != full["test"])]
    low = x[x["balanced_acc"] < 0.35]
    if len(low):
        pairs = sorted({f"{a}->{b}" for a, b in zip(low["train"], low["test"])})
        print("\n[WARNING] Balanced accuracy far below 0.5 for:", ", ".join(pairs))
    agree = x.dropna(subset=["overlap_label_agreement"]).groupby(["train", "test"])["overlap_label_agreement"].mean()
    for (a, b), v in agree.items():
        verdict = "labels consistent" if v > 0.9 else ("LABELS LOOK INVERTED -> try --welfake-flip" if v < 0.1 else "mixed")
        print(f"[overlap check] same articles in {a} and {b}: labels agree {v*100:.1f}% -> {verdict}")

    print(f"\nSaved CSVs and heatmaps to: {args.out_dir.resolve()}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase8_cross_dataset_dl.py
"""
phase8_cross_dataset_dl.py
==========================
Deep-learning version of the cross-dataset experiment with SEVERAL model families.
Each model is trained on one dataset and tested on the datasets it has never seen.
It imports phase7_cross_dataset.py, so the data handling is identical to the classical run:

    * same loaders, same label convention (1 = True/real, 0 = Fake)
    * articles shared between datasets are removed BEFORE any split
    * same 80/20 in-domain split per seed, same metrics (accuracy, macro-F1,
      balanced accuracy, majority baseline, share predicted "True")

Models (choose with --models, comma-separated)
----------------------------------------------
  hf:<name>            any pretrained Hugging Face encoder, e.g.
                       hf:distilbert-base-uncased   (66M params, default)
                       hf:bert-base-uncased         (110M)
                       hf:roberta-base              (125M, about 2.5x slower than DistilBERT)
  bilstm               bidirectional LSTM trained from scratch (embedding 128, hidden 128)
  textcnn              CNN with filter widths 3/4/5 trained from scratch (Kim 2014 style)
  compact_transformer  small transformer encoder trained from scratch, INSPIRED by the
                       reference paper (sinusoidal positions, GELU, label smoothing 0.1).
                       It is NOT an exact replication: the paper's full spec and training
                       setup are not reproduced here.

The from-scratch models use the bert-base-uncased tokenizer for their vocabulary and
start with randomly initialised embeddings (no pretraining). That difference between
"pretrained" and "from scratch" is itself part of what the experiment tests.

Two text modes (the DL counterpart of phase7's "basic" / "delaked"):
    raw      = the article exactly as written (the model can see "(Reuters)", URLs, ...)
    delaked  = URLs and leakage terms removed, normal words and punctuation kept
    strict   = delaked + dateline removed + ALL punctuation removed (letters/digits only)

Needs a GPU for practical speed (Kaggle: Settings -> Accelerator -> GPU) and internet once
(pretrained weights + tokenizer download). Both scripts must sit in the same folder:

    python phase8_cross_dataset_dl.py --isot-fake Fake.csv --isot-true True.csv \\
        --welfake WELFake_Dataset.csv --welfake-flip --liar-csv liar_binary.csv

Quick smoke run (a few minutes):
    ... --max-train 3000 --max-eval 3000 --epochs 1 --mode delaked
"""
from __future__ import annotations

import argparse
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split

import phase7_cross_dataset as p7

CHAR_LIMIT = 2000                      # 256 tokens is roughly 1,500 characters
SCRATCH_TOKENIZER = "bert-base-uncased"
DEFAULT_MODELS = "hf:distilbert-base-uncased,bilstm,textcnn,compact_transformer"

# (epochs, learning rate) used when --epochs / --lr are not given
HPARAMS = {
    "hf": (2, 2e-5),
    "bilstm": (4, 1e-3),
    "textcnn": (4, 1e-3),
    "compact_transformer": (5, 5e-4),
}


def kind_of(name: str) -> str:
    return "hf" if name.startswith("hf:") else name


def short_name(name: str) -> str:
    return name[3:] if name.startswith("hf:") else name


def hparams(name: str, args) -> tuple[int, float]:
    if kind_of(name) not in HPARAMS:
        raise SystemExit(f"Unknown model '{name}'. Use hf:<name>, bilstm, textcnn or compact_transformer.")
    epochs, lr = HPARAMS[kind_of(name)]
    return (args.epochs or epochs), (args.lr or lr)


# --------------------------------------------------------------------------
# Text preparation
# --------------------------------------------------------------------------
# "CITY (Reuters) -" at the start of an article, e.g. "WASHINGTON (Reuters) - ..."
DATELINE = re.compile(r"^\s*[A-Z][A-Za-z .,'/&-]{1,60}?\s*\((?:Reuters|AP|AFP)[^)]{0,20}\)\s*[-\u2013\u2014]*\s*")


def prepare_text(text: str, mode: str) -> str:
    text = str(text)[:CHAR_LIMIT]
    if mode == "raw":
        return text
    if mode == "delaked":
        # phase7 cleaning only; punctuation is kept, so formatting leftovers such as "( ) -" survive
        return p7.remove_leakage_terms(p7.strip_urls(text))
    # "strict": also drop the dateline and ALL punctuation / symbols / typographic quotes,
    # keeping only lowercase letters, digits and spaces (stop-words stay, unlike phase7's classical cleaning)
    text = DATELINE.sub("", text)
    cleaned = p7.remove_leakage_terms(p7.strip_urls(text))
    cleaned = re.sub(r"\b21st century\b", " ", cleaned)           # remaining outlet name in ISOT's fake articles
    cleaned = re.sub(r"[^a-z0-9\s]", " ", cleaned)
    return re.sub(r"\s+", " ", cleaned).strip()


def cap(df: pd.DataFrame, n: int | None, seed: int) -> pd.DataFrame:
    """Stratified random subsample to about n rows (keeps the class balance)."""
    if not n or len(df) <= n:
        return df
    return df.groupby("label", group_keys=False).sample(frac=n / len(df), random_state=seed)


# --------------------------------------------------------------------------
# Torch / transformers parts (imported lazily so the rest can be tested without a GPU)
# --------------------------------------------------------------------------
def get_device():
    import torch
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


_TOKENIZERS: dict = {}


def get_tokenizer(model_name: str):
    from transformers import AutoTokenizer
    key = model_name[3:] if model_name.startswith("hf:") else SCRATCH_TOKENIZER
    if key not in _TOKENIZERS:
        _TOKENIZERS[key] = AutoTokenizer.from_pretrained(key)
    return _TOKENIZERS[key]


def set_seed(seed: int) -> None:
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


_SCRATCH: dict = {}


def _scratch_classes() -> dict:
    """Models trained from scratch. Each forward() mimics Hugging Face: returns .loss and .logits."""
    if _SCRATCH:
        return _SCRATCH
    import math
    from types import SimpleNamespace

    import torch
    import torch.nn as nn

    def finish(logits, labels, smoothing: float = 0.0):
        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(logits.float(), labels, label_smoothing=smoothing)
        return SimpleNamespace(loss=loss, logits=logits)

    def masked_mean(h, attention_mask):
        m = attention_mask.unsqueeze(-1).to(h.dtype)
        return (h * m).sum(1) / m.sum(1).clamp(min=1)

    class BiLSTM(nn.Module):
        def __init__(self, vocab, pad_id, emb=128, hid=128):
            super().__init__()
            self.emb = nn.Embedding(vocab, emb, padding_idx=pad_id)
            self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
            self.drop = nn.Dropout(0.3)
            self.out = nn.Linear(2 * hid, 2)

        def forward(self, input_ids, attention_mask=None, labels=None, **kw):
            lengths = attention_mask.sum(1).clamp(min=1).cpu()
            x = self.drop(self.emb(input_ids))
            packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
            h, _ = self.lstm(packed)
            h, _ = nn.utils.rnn.pad_packed_sequence(h, batch_first=True, total_length=input_ids.size(1))
            return finish(self.out(self.drop(masked_mean(h, attention_mask))), labels)

    class TextCNN(nn.Module):
        def __init__(self, vocab, pad_id, emb=128, filters=100, widths=(3, 4, 5)):
            super().__init__()
            self.widths = widths
            self.emb = nn.Embedding(vocab, emb, padding_idx=pad_id)
            self.convs = nn.ModuleList([nn.Conv1d(emb, filters, k) for k in widths])
            self.drop = nn.Dropout(0.5)
            self.out = nn.Linear(filters * len(widths), 2)

        def forward(self, input_ids, attention_mask=None, labels=None, **kw):
            x = self.emb(input_ids).transpose(1, 2)                # batch, emb, length
            if x.size(2) < max(self.widths):                       # very short batches
                x = nn.functional.pad(x, (0, max(self.widths) - x.size(2)))
            feats = [torch.relu(c(x)).max(dim=2).values for c in self.convs]
            return finish(self.out(self.drop(torch.cat(feats, dim=1))), labels)

    class CompactTransformer(nn.Module):
        def __init__(self, vocab, pad_id, d=192, heads=6, layers=4, ff=384, max_len=512, drop=0.1):
            super().__init__()
            self.d = d
            self.emb = nn.Embedding(vocab, d, padding_idx=pad_id)
            pe = torch.zeros(max_len, d)
            pos = torch.arange(max_len).unsqueeze(1).float()
            div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
            pe[:, 0::2] = torch.sin(pos * div)
            pe[:, 1::2] = torch.cos(pos * div)
            self.register_buffer("pe", pe)
            layer = nn.TransformerEncoderLayer(d, heads, ff, drop, activation="gelu", batch_first=True)
            self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
            self.norm = nn.LayerNorm(d)
            self.drop = nn.Dropout(drop)
            self.out = nn.Linear(d, 2)

        def forward(self, input_ids, attention_mask=None, labels=None, **kw):
            x = self.emb(input_ids) * math.sqrt(self.d) + self.pe[: input_ids.size(1)]
            h = self.enc(self.drop(x), src_key_padding_mask=(attention_mask == 0))
            pooled = self.norm(masked_mean(h, attention_mask))
            return finish(self.out(self.drop(pooled)), labels, smoothing=0.1)

    _SCRATCH.update(bilstm=BiLSTM, textcnn=TextCNN, compact_transformer=CompactTransformer)
    return _SCRATCH


def build_model(name: str, tok):
    if name.startswith("hf:"):
        from transformers import AutoModelForSequenceClassification
        return AutoModelForSequenceClassification.from_pretrained(name[3:], num_labels=2)
    classes = _scratch_classes()
    if name not in classes:
        raise SystemExit(f"Unknown model '{name}'.")
    return classes[name](len(tok), tok.pad_token_id)


def train_model(model_name: str, texts: list[str], labels: list[int], tok, device, args, seed: int):
    """Train a fresh model on (texts, labels). Returns the trained model."""
    import torch
    from torch.utils.data import DataLoader
    from transformers import get_linear_schedule_with_warmup

    epochs, lr = hparams(model_name, args)
    set_seed(seed)
    use_amp = device.type == "cuda"
    model = build_model(model_name, tok).to(device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"    {short_name(model_name)}: {n_params:.1f}M parameters, {epochs} epochs, lr {lr}", flush=True)

    def collate(batch_idx):
        enc = tok([texts[i] for i in batch_idx], truncation=True, max_length=args.max_len,
                  padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor([labels[i] for i in batch_idx])
        return enc

    gen = torch.Generator()
    gen.manual_seed(seed)
    loader = DataLoader(list(range(len(texts))), batch_size=args.batch_size, shuffle=True,
                        collate_fn=collate, generator=gen)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * total_steps), total_steps)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except Exception:  # older torch
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    model.train()
    step, t0, running = 0, time.time(), 0.0
    for epoch in range(epochs):
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                loss = model(**batch).loss
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running += loss.item()
            step += 1
            if step % 100 == 0:
                print(f"    epoch {epoch+1}/{epochs} step {step}/{total_steps} "
                      f"loss {running/100:.4f}  ({time.time()-t0:.0f}s)", flush=True)
                running = 0.0
    return model


def predict(model, tok, texts: list[str], device, args) -> np.ndarray:
    """Predicted labels (0/1) for texts, in the original order."""
    import torch

    use_amp = device.type == "cuda"
    model.eval()
    order = np.argsort([len(t) for t in texts])          # similar lengths together = less padding
    preds = np.zeros(len(texts), dtype=int)
    with torch.no_grad():
        for i in range(0, len(order), args.eval_batch_size):
            ids = order[i:i + args.eval_batch_size]
            enc = tok([texts[j] for j in ids], truncation=True, max_length=args.max_len,
                      padding=True, return_tensors="pt").to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(**enc).logits
            preds[ids] = logits.argmax(dim=-1).cpu().numpy()
    return preds


def free_model(model) -> None:
    try:
        import gc
        import torch
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


# --------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------
def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--isot-fake", type=Path)
    ap.add_argument("--isot-true", type=Path)
    ap.add_argument("--welfake", type=Path)
    ap.add_argument("--welfake-flip", action="store_true")
    ap.add_argument("--liar-dir", type=Path)
    ap.add_argument("--liar-csv", type=Path)
    ap.add_argument("--no-dedup-across", action="store_true")
    ap.add_argument("--mode", choices=["raw", "delaked", "strict", "both", "all"], default="both",
                    help="both = raw + delaked; all = raw + delaked + strict")
    ap.add_argument("--models", default=DEFAULT_MODELS, help="comma-separated, see top of file")
    ap.add_argument("--epochs", type=int, default=None, help="override epochs for ALL models")
    ap.add_argument("--lr", type=float, default=None, help="override learning rate for ALL models")
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--eval-batch-size", type=int, default=128)
    ap.add_argument("--max-len", type=int, default=256)
    ap.add_argument("--max-train", type=int, default=20000, help="cap on training rows per run")
    ap.add_argument("--max-eval", type=int, default=20000, help="cap on rows per test set")
    ap.add_argument("--max-per-dataset", type=int, default=None, help="cap each dataset before everything else")
    ap.add_argument("--seeds", default="42", help="comma-separated; each extra seed multiplies the run time")
    ap.add_argument("--out-dir", type=Path, default=Path("cross_dataset_results_dl"))
    args = ap.parse_args()

    seeds = [int(s) for s in args.seeds.split(",")]
    model_names = [m.strip() for m in args.models.split(",") if m.strip()]
    for m in model_names:
        hparams(m, args)                                  # fail early on a typo
    args.out_dir.mkdir(parents=True, exist_ok=True)

    # ---- load + dedup exactly like phase7 -------------------------------------------------
    raw: dict[str, pd.DataFrame] = {}
    if args.isot_fake and args.isot_true:
        raw["ISOT"] = p7.finalize(p7.load_isot(args.isot_fake, args.isot_true), "isot")
    if args.welfake:
        raw["WELFake"] = p7.finalize(p7.load_welfake(args.welfake, args.welfake_flip), "welfake")
    if args.liar_dir:
        raw["LIAR"] = p7.finalize(p7.load_liar(args.liar_dir), "liar")
    elif args.liar_csv:
        raw["LIAR"] = p7.finalize(p7.load_liar_csv(args.liar_csv), "liar")
    if len(raw) < 2:
        raise SystemExit("Need at least 2 datasets.")
    if not args.no_dedup_across:
        print("\n=== Removing articles shared between datasets (before any split) ===")
        p7.dedup_across(raw)
    raw = {n: p7.subsample(d, args.max_per_dataset) for n, d in raw.items()}

    print("\n=== Datasets ===")
    for n, d in raw.items():
        print(f"{n:8s} rows={len(d):6d}  true={d['label'].mean()*100:5.1f}%")

    device = get_device()
    print(f"\ndevice: {getattr(device, 'type', device)} | models: {model_names} | max_len: {args.max_len} "
          f"| max_train: {args.max_train} | seeds: {seeds}")

    names = list(raw)
    modes = {"both": ["raw", "delaked"], "all": ["raw", "delaked", "strict"]}.get(args.mode, [args.mode])
    all_rows: list[dict] = []

    for mode in modes:
        t0 = time.time()
        prepared = {}
        for n, d in raw.items():
            d = d.copy()
            d["x"] = d["text"].map(lambda t: prepare_text(t, mode))
            prepared[n] = d[d["x"].str.len() > 0].reset_index(drop=True)
        print(f"\n=== Text prepared ({mode}) in {time.time()-t0:.0f}s ===")

        mode_rows: list[dict] = []
        for model_name in model_names:
            label = short_name(model_name)
            tok = get_tokenizer(model_name)
            for src in names:
                df = prepared[src]
                for seed in seeds:
                    tr_idx, te_idx = train_test_split(
                        np.arange(len(df)), test_size=0.2, random_state=seed, stratify=df["label"])
                    train = cap(df.iloc[tr_idx], args.max_train, seed)
                    own_test = df.iloc[te_idx]
                    print(f"\n[{mode}] {label}: training on {src} (seed {seed}), {len(train)} articles", flush=True)
                    model = train_model(model_name, train["x"].tolist(), train["label"].astype(int).tolist(),
                                        tok, device, args, seed)
                    for tgt in names:
                        test = own_test if tgt == src else prepared[tgt]
                        test = cap(test, args.max_eval, 0)
                        pred = predict(model, tok, test["x"].tolist(), device, args)
                        y = test["label"].to_numpy().astype(int)
                        row = dict(
                            mode=mode, train=src, test=tgt, model=label, seed=seed, n_test=len(test),
                            accuracy=accuracy_score(y, pred), balanced_acc=balanced_accuracy_score(y, pred),
                            macro_f1=f1_score(y, pred, average="macro"),
                            pred_true_rate=float(pred.mean()), actual_true_rate=float(y.mean()),
                            majority_baseline=float(max(y.mean(), 1 - y.mean())),
                        )
                        mode_rows.append(row)
                        print(f"    -> tested on {tgt:8s} n={len(test):6d} acc={row['accuracy']*100:5.2f} "
                              f"bal_acc={row['balanced_acc']*100:5.2f} predicted-true={row['pred_true_rate']*100:5.1f}%",
                              flush=True)
                    free_model(model)
                    pd.DataFrame(all_rows + mode_rows).to_csv(args.out_dir / "results_dl_partial.csv", index=False)

        res = pd.DataFrame(mode_rows)
        res.to_csv(args.out_dir / f"results_dl_{mode}.csv", index=False)
        all_rows.extend(mode_rows)

        for model_name in model_names:
            label = short_name(model_name)
            acc_m, acc_s = p7.pivot(res, label, "accuracy", names)
            f1_m, f1_s = p7.pivot(res, label, "macro_f1", names)
            bal_m, bal_s = p7.pivot(res, label, "balanced_acc", names)
            print(f"\n--- {mode} | {label} | accuracy % (mean +- std over {len(seeds)} seed(s)) ---")
            print(p7.fmt_table(acc_m, acc_s).to_string())
            print(f"--- {mode} | {label} | macro-F1 % ---")
            print(p7.fmt_table(f1_m, f1_s).to_string())
            print(f"--- {mode} | {label} | balanced accuracy % (50 = constant guess) ---")
            print(p7.fmt_table(bal_m, bal_s).to_string())
            p7.heatmap(acc_m, f"{label} ({mode} text)", args.out_dir / f"matrix_dl_{mode}_{label.replace('/', '_')}.png")

        diag = (res[res["train"] != res["test"]]
                .groupby(["model", "train", "test"])[["accuracy", "balanced_acc", "majority_baseline",
                                                      "pred_true_rate", "actual_true_rate"]].mean().round(3))
        print(f"\n--- {mode} | cross-dataset diagnostics (all models) ---")
        print(diag.to_string())

    full = pd.DataFrame(all_rows)
    full.to_csv(args.out_dir / "results_dl_all.csv", index=False)

    print("\n=== Summary: ISOT <-> WELFake only (the two news-article datasets), accuracy % ===")
    news = full[full["train"].isin(["ISOT", "WELFake"]) & full["test"].isin(["ISOT", "WELFake"])]
    if not news.empty:
        news = news.assign(kind=np.where(news["train"] == news["test"], "in-domain", "cross-dataset"))
        summ = news.groupby(["mode", "model", "kind"])["accuracy"].mean().unstack("kind") * 100
        if {"in-domain", "cross-dataset"} <= set(summ.columns):
            summ["gap"] = summ["in-domain"] - summ["cross-dataset"]
        print(summ.round(2).to_string())
    print(f"\nSaved CSVs and heatmaps to: {args.out_dir.resolve()}")


if __name__ == "__main__":
    main()


## 4. See what each text mode does to an article (instant)
Check that the `strict` output has no brackets, dashes or quote characters left.

In [ ]:
import sys; sys.path.insert(0, '/kaggle/working')
import phase8_cross_dataset_dl as p8
sample = "WASHINGTON (Reuters) - The U.S. Senate said on Tuesday it\u2019s \u201cready\u201d to vote \u2013 see https://www.reuters.com/x for more."
for m in ['raw', 'delaked', 'strict']:
    print(m.upper(), '->', p8.prepare_text(sample, m))

## 5. Main run: strict text, 3 seeds, DistilBERT and the compact transformer
Expect roughly 45-60 minutes on a T4 (rough estimate). Use **Save Version -> Save & Run All (Commit)** to run in the background.

In [ ]:
!python /kaggle/working/phase8_cross_dataset_dl.py --isot-fake {FAKE} --isot-true {TRUE} \
  --welfake {WEL} --welfake-flip --liar-csv /kaggle/working/liar_binary.csv \
  --mode strict --models hf:distilbert-base-uncased,compact_transformer --seeds 42,43,44 \
  --out-dir /kaggle/working/results_dl_strict_seeds

## 6. Optional: all four models on strict text, 1 seed (about 25-30 minutes)
Run this only after cell 5 works. It lets you compare BiLSTM and TextCNN under the same cleaning.

In [ ]:
!python /kaggle/working/phase8_cross_dataset_dl.py --isot-fake {FAKE} --isot-true {TRUE} \
  --welfake {WEL} --welfake-flip --liar-csv /kaggle/working/liar_binary.csv \
  --mode strict --out-dir /kaggle/working/results_dl_strict_all

## 7. Compact results table (run after cell 5 finishes)

In [ ]:
import pandas as pd
r = pd.read_csv('/kaggle/working/results_dl_strict_seeds/results_dl_all.csv')
print('rows:', len(r))
for (mode, model), g in r.groupby(['mode', 'model']):
    acc = g.pivot_table(index='train', columns='test', values='accuracy', aggfunc=['mean','std']) * 100
    m, s = acc['mean'].round(1), acc['std'].round(1)
    print(f'\n{mode} | {model} | accuracy % (mean +- std over seeds)')
    print((m.astype(str) + ' +- ' + s.astype(str)).to_string())